# DeepLabV3 (`seg-deeplabv3`)
Model: **DeepLabV3 with a ResNet-50 backbone** (COCO-pretrained), fine-tuned for **3-class** UVFD

Dataset: [Kaggle: https://www.kaggle.com/datasets/samirhossain2001/automatic-hair-removal]

## 1. Setup & imports

In [ ]:
from __future__ import annotations
import os, re, glob, json, math, time, random, warnings
from collections import defaultdict
from copy import deepcopy
from typing import Callable

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch, torch.nn as nn, torch.nn.functional as F, torchvision
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights

try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "albumentations"], check=True)
    import albumentations as A
    from albumentations.pytorch import ToTensorV2

try:
    from tqdm.auto import tqdm
except ModuleNotFoundError:
    def tqdm(x, **k): return x

warnings.filterwarnings("ignore")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
for name, mod in [("torch", torch), ("torchvision", torchvision), ("albumentations", A)]:
    print(f"{name:14}: {mod.__version__}")
print("device        :", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU only")

## 2. CONFIG

In [ ]:
def find_data_root() -> str:
    candidates = [
        "/kaggle/input/datasets/samirhossain2001/automatic-hair-removal/Dataset",
        "e:/CSE438_DIP/CSE438_Assignment/Dataset",
    ]
    candidates += [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/black_masks", recursive=True)]
    for root in candidates:
        if root and all(os.path.isdir(os.path.join(root, d)) for d in ("images", "black_masks", "white_masks")):
            return root
    raise FileNotFoundError("No folder with images/ black_masks/ white_masks/ found.")

DATA_ROOT = find_data_root()
CONFIG = {
    "DATA_ROOT":    DATA_ROOT,
    "IMAGES_DIR":   os.path.join(DATA_ROOT, "images"),
    "BLACK_DIR":    os.path.join(DATA_ROOT, "black_masks"),
    "WHITE_DIR":    os.path.join(DATA_ROOT, "white_masks"),
    "OUT_DIR":      "/kaggle/working" if os.path.isdir("/kaggle/working") else ".",
    "MODEL_NAME":   "deeplabv3_resnet50",
    "SEED":         42,
    "IMG_SIZE":     512,
    "NUM_CLASSES":  3,
    "CLASS_NAMES":  {0: "background", 1: "dark_hair_ruler", 2: "light_hair_uv"},
    "CLASS_COLORS": {1: (255, 0, 0), 2: (0, 255, 0)},
    "MASK_THRESH":  127,
    "SPLIT_RATIO":  (0.70, 0.15, 0.15),
    "BATCH_SIZE":   8,
    "EPOCHS":       80,
    "LR_BACKBONE":  1e-5,
    "LR_HEAD":      1e-4,
    "WEIGHT_DECAY": 1e-4,
    "WARMUP_FRAC":  0.05,
    "AUX_WEIGHT":   0.4,
    "NUM_WORKERS":  2,
    "USE_AMP":      True,
    "WEIGHT_TYPE":  "inverse_freq",   
    "DICE_INCLUDE_BG": False,        
}
for k, v in CONFIG.items():
    print(f"{k:12}: {v}")

## 3. Shared utilities

### 3.1 Data helpers

In [ ]:
def set_seed(seed: int = CONFIG["SEED"]) -> None:
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def group_key(filename: str) -> str:
    stem = filename[:-4] if filename.lower().endswith(".png") else filename
    stem = re.sub(r"^crop_\d+_", "", stem)
    stem = re.sub(r"_crop_\d+$", "", stem)
    return stem

def load_rgb(filename: str, size: int | None = None) -> np.ndarray:
    size = size or CONFIG["IMG_SIZE"]
    return np.asarray(Image.open(os.path.join(CONFIG["IMAGES_DIR"], filename)).convert("RGB").resize((size, size)))

def _mask(folder: str, filename: str, size: int) -> np.ndarray:
    m = Image.open(os.path.join(folder, filename)).convert("L").resize((size, size), Image.NEAREST)
    return np.asarray(m) > CONFIG["MASK_THRESH"]

def build_label(filename: str, size: int | None = None) -> np.ndarray:
    size = size or CONFIG["IMG_SIZE"]
    label = np.zeros((size, size), np.uint8)
    if filename in BLACK_MASKS: label[_mask(CONFIG["BLACK_DIR"], filename, size)] = 1
    if filename in WHITE_MASKS: label[_mask(CONFIG["WHITE_DIR"], filename, size)] = 2
    return label

def overlay(image: np.ndarray, label: np.ndarray, alpha: float = 0.5) -> np.ndarray:
    out = image.astype(np.float32).copy()
    for cls, colour in CONFIG["CLASS_COLORS"].items():
        m = label == cls
        for c in range(3):
            out[..., c][m] = (1 - alpha) * out[..., c][m] + alpha * colour[c]
    return out.astype(np.uint8)

set_seed()
BLACK_MASKS = set(os.listdir(CONFIG["BLACK_DIR"]))
IMAGE_FILES = sorted(os.listdir(CONFIG["IMAGES_DIR"]))
WHITE_MASKS = {f for f in os.listdir(CONFIG["WHITE_DIR"]) if f in set(IMAGE_FILES)}
print("images:", len(IMAGE_FILES))

### 3.2 Dataset & augmentation

In [ ]:
IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

def build_transforms(img_size: int) -> tuple[A.Compose, A.Compose]:
    train_tf = A.Compose([
        A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5),
        A.Affine(scale=(0.9, 1.1), translate_percent=(0.0, 0.05), rotate=(-20, 20), mode=0, p=0.3),
        A.RandomBrightnessContrast(0.2, 0.2, p=0.3),
        A.OneOf([A.CLAHE(2.0), A.RandomGamma((80, 120))], p=0.2),
        A.ElasticTransform(alpha=1.0, sigma=50.0, p=0.15),
        A.Normalize(IMAGENET_MEAN, IMAGENET_STD), ToTensorV2(),
    ])
    eval_tf = A.Compose([A.Normalize(IMAGENET_MEAN, IMAGENET_STD), ToTensorV2()])
    return train_tf, eval_tf

class SegDataset(torch.utils.data.Dataset):
    def __init__(self, files: list[str], transform: A.Compose):
        self.files, self.transform = files, transform
    def __len__(self) -> int:
        return len(self.files)
    def __getitem__(self, i: int):
        fn = self.files[i]
        out = self.transform(image=load_rgb(fn), mask=build_label(fn))
        return out["image"], out["mask"].long(), fn

train_tf, eval_tf = build_transforms(CONFIG["IMG_SIZE"])
print("transforms ready | train ops:", len(train_tf.transforms), "| eval ops:", len(eval_tf.transforms))

### 3.3 Metrics

In [ ]:
class ConfusionMatrix:
    def __init__(self, num_classes: int, device: str = "cpu"):
        self.k = num_classes
        self.mat = torch.zeros(num_classes, num_classes, dtype=torch.long, device=device)

    @torch.no_grad()
    def update(self, pred: torch.Tensor, target: torch.Tensor) -> None:
        idx = target.reshape(-1) * self.k + pred.reshape(-1)
        self.mat += torch.bincount(idx, minlength=self.k ** 2).reshape(self.k, self.k)

    def _tp_fp_fn(self):
        cm = self.mat.double(); tp = cm.diag()
        return tp, cm.sum(0) - tp, cm.sum(1) - tp, cm

    @property
    def per_class_iou(self) -> np.ndarray:
        tp, fp, fn, _ = self._tp_fp_fn()
        return (tp / (tp + fp + fn).clamp(min=1e-9)).cpu().numpy()

    @property
    def per_class_dice(self) -> np.ndarray:
        tp, fp, fn, _ = self._tp_fp_fn()
        return (2 * tp / (2 * tp + fp + fn).clamp(min=1e-9)).cpu().numpy()

    @property
    def mean_iou(self) -> float:  return float(self.per_class_iou.mean())
    @property
    def mean_dice(self) -> float: return float(self.per_class_dice.mean())
    @property
    def foreground_mean_iou(self) -> float:
        return float(self.per_class_iou[1:].mean())

    @property
    def pixel_accuracy(self) -> float:
        tp, _, _, cm = self._tp_fp_fn()
        return float(tp.sum() / cm.sum().clamp(min=1e-9))

    @property
    def mean_pixel_accuracy(self) -> float:
        tp, _, _, cm = self._tp_fp_fn()
        return float((tp / cm.sum(1).clamp(min=1e-9)).mean())

    @property
    def matrix(self) -> np.ndarray:
        return self.mat.cpu().numpy()

@torch.no_grad()
def evaluate(model: nn.Module, loader, criterion=None) -> tuple[ConfusionMatrix, float]:
    model.eval()
    cm = ConfusionMatrix(CONFIG["NUM_CLASSES"], DEVICE); total_loss = n = 0
    for images, masks, _ in loader:
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        out = model(images)
        cm.update(out["out"].argmax(1), masks)
        if criterion is not None:
            total_loss += criterion(out, masks).item() * images.size(0); n += images.size(0)
    return cm, (total_loss / max(n, 1))
print("ConfusionMatrix + evaluate() ready")

## 4. Load the shared split & class weights from NB0

In [ ]:
def find_artifact(name: str) -> str | None:
    for base in ["/kaggle/working", "."] + glob.glob("/kaggle/input/*") + glob.glob("/kaggle/input/**/", recursive=True):
        path = os.path.join(base, name)
        if os.path.isfile(path): return path
    return None

def make_grouped_split(files: list[str], seed: int, ratio: tuple[float, float, float]) -> dict[str, list[str]]:
    groups: dict[str, list[str]] = defaultdict(list)
    for f in files: groups[group_key(f)].append(f)
    sized = [(g, len(m)) for g, m in groups.items()]; random.Random(seed).shuffle(sized)
    total = len(files); target = dict(zip(("train", "val", "test"), (r * total for r in ratio)))
    filled = {"train": 0, "val": 0, "test": 0}; assign = {}
    for g, size in sorted(sized, key=lambda gs: -gs[1]):
        sp = max(target, key=lambda k: target[k] - filled[k]); assign[g] = sp; filled[sp] += size
    split = {"train": [], "val": [], "test": []}
    for g in sorted(groups): split[assign[g]].extend(sorted(groups[g]))
    return split

split_path, weights_path = find_artifact("split.json"), find_artifact("class_weights.json")
if split_path:
    split = {k: json.load(open(split_path))[k] for k in ("train", "val", "test")}
    print("loaded split.json from", split_path)
else:
    warnings.warn("split.json not found - regenerating the identical split (seed 42).")
    split = make_grouped_split(IMAGE_FILES, CONFIG["SEED"], CONFIG["SPLIT_RATIO"])

if weights_path:
    class_weights = json.load(open(weights_path)); print("loaded class_weights.json from", weights_path)
else:
    warnings.warn("class_weights.json not found - recomputing from the TRAIN split only.")
    counts = np.zeros(CONFIG["NUM_CLASSES"], np.int64)
    for fn in split["train"]: counts += np.bincount(build_label(fn).ravel(), minlength=CONFIG["NUM_CLASSES"])
    frac = counts / counts.sum(); inv = 1 / frac; inv /= inv.sum() / CONFIG["NUM_CLASSES"]
    class_weights = {"inverse_freq": inv.tolist(), "median_freq": (np.median(frac) / frac).tolist()}

seen = set()
for k in ("train", "val", "test"):
    s = set(split[k]); assert not (s & seen), "leak: image shared across splits"; seen |= s
print({k: len(v) for k, v in split.items()},
      "| CE weights:", [round(x, 3) for x in class_weights[CONFIG["WEIGHT_TYPE"]]])

### DataLoaders

In [ ]:
def seed_worker(worker_id: int) -> None:
    worker_seed = (torch.initial_seed() + worker_id) % 2**32
    np.random.seed(worker_seed); random.seed(worker_seed)

loader_gen = torch.Generator().manual_seed(CONFIG["SEED"])
common = dict(batch_size=CONFIG["BATCH_SIZE"], num_workers=CONFIG["NUM_WORKERS"],
              pin_memory=(DEVICE == "cuda"), worker_init_fn=seed_worker,
              persistent_workers=(CONFIG["NUM_WORKERS"] > 0))

set_seed()
loaders = {
    "train": torch.utils.data.DataLoader(SegDataset(split["train"], train_tf),
                 shuffle=True, drop_last=True, generator=loader_gen, **common),
    "val":   torch.utils.data.DataLoader(SegDataset(split["val"], eval_tf), shuffle=False, **common),
    "test":  torch.utils.data.DataLoader(SegDataset(split["test"], eval_tf), shuffle=False, **common),
}
print({k: f"{len(v.dataset)} imgs / {len(v)} batches" for k, v in loaders.items()})

## Visualize the data & augmentation pipeline

### Visualization helpers

In [ ]:
def denormalize(image_tensor: torch.Tensor) -> np.ndarray:
    x = image_tensor.permute(1, 2, 0).cpu().numpy() * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    return (np.clip(x, 0, 1) * 255).astype(np.uint8)

def plot_grid(columns: list[tuple[str, list[np.ndarray]]], row_labels: list[str], suptitle: str,
              col_w: float = 2.6, row_h: float = 2.6) -> None:
    nrows, ncols = len(row_labels), len(columns)
    fig, ax = plt.subplots(nrows, ncols, figsize=(col_w * ncols, row_h * nrows), squeeze=False)
    for j, (caption, imgs) in enumerate(columns):
        for i, im in enumerate(imgs):
            ax[i, j].imshow(im); ax[i, j].set_xticks([]); ax[i, j].set_yticks([])
            if i == 0: ax[i, j].set_title(caption, fontsize=8)
    for i, label in enumerate(row_labels):
        ax[i, 0].set_ylabel(label, fontsize=9)
    fig.suptitle(suptitle, y=1.01); fig.tight_layout(); plt.show()

print("denormalize + plot_grid ready")

### Augmentation showcase

In [ ]:
def build_showcase(filename: str):
    img, mask = load_rgb(filename), build_label(filename)
    single = [
        ("original",           None),
        ("HorizontalFlip",     A.HorizontalFlip(p=1.0)),
        ("VerticalFlip",       A.VerticalFlip(p=1.0)),
        ("RandomRotate90",     A.RandomRotate90(p=1.0)),
        ("Affine",             A.Affine(scale=(0.9, 1.1), translate_percent=(0.0, 0.05), rotate=(-20, 20), mode=0, p=1.0)),
        ("BrightnessContrast", A.RandomBrightnessContrast(0.2, 0.2, p=1.0)),
        ("CLAHE",              A.CLAHE(clip_limit=2.0, p=1.0)),
        ("RandomGamma",        A.RandomGamma((80, 120), p=1.0)),
        ("ElasticTransform",   A.ElasticTransform(alpha=1.0, sigma=50.0, p=1.0)),
    ]
    columns = []
    for name, tf in single:
        out = {"image": img, "mask": mask} if tf is None else A.Compose([tf])(image=img, mask=mask)
        columns.append((name, [out["image"], overlay(out["image"], out["mask"])]))
    return columns

set_seed()
demo_file = next(f for f in split["train"] if f in BLACK_MASKS and f in WHITE_MASKS)
plot_grid(build_showcase(demo_file), ["image", "overlay"], "Augmentation showcase - one transform each")

### Full-pipeline variety

In [ ]:
display_tf = A.Compose(train_tf.transforms[:-2]) 

def augment_variety(filename: str, n: int = 6):
    img, mask = load_rgb(filename), build_label(filename)
    cols = [("original", [img, overlay(img, mask)])]
    for i in range(n):
        out = display_tf(image=img, mask=mask)
        cols.append((f"aug #{i + 1}", [out["image"], overlay(out["image"], out["mask"])]))
    return cols

set_seed()
plot_grid(augment_variety(demo_file, n=6), ["image", "overlay"], "Full pipeline: 6 random augmentations of one image")

### Post-augmentation training batch 

In [ ]:
set_seed()
train_ds = loaders["train"].dataset
batch = [train_ds[i] for i in random.sample(range(len(train_ds)), 6)]
cols = [(fn[:14], [denormalize(img), overlay(denormalize(img), mask.numpy())]) for img, mask, fn in batch]
plot_grid(cols, ["augmented", "overlay"], "Post-augmentation training batch")

## 5. Model: DeepLabV3-ResNet50 (COCO-pretrained)

In [ ]:
def build_model() -> nn.Module:
    model = deeplabv3_resnet50(weights=DeepLabV3_ResNet50_Weights.DEFAULT, aux_loss=True)
    model.classifier[4]     = nn.Conv2d(256, CONFIG["NUM_CLASSES"], kernel_size=1)
    model.aux_classifier[4] = nn.Conv2d(256, CONFIG["NUM_CLASSES"], kernel_size=1)
    return model

model = build_model().to(DEVICE)
print(f"DeepLabV3-ResNet50 | params: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M | "
      f"heads -> {model.classifier[4].out_channels} classes")

## 6. Loss, optimizer & LR schedule

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, num_classes: int, include_background: bool = True, eps: float = 1.0):
        super().__init__(); self.k, self.include_bg, self.eps = num_classes, include_background, eps
    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        prob = F.softmax(logits, dim=1)
        onehot = F.one_hot(target, self.k).permute(0, 3, 1, 2).float()
        dims = (0, 2, 3)
        inter = (prob * onehot).sum(dims); card = prob.sum(dims) + onehot.sum(dims)
        dice = (2 * inter + self.eps) / (card + self.eps)          # per-class Dice
        if not self.include_bg: dice = dice[1:]                    # drop background (class 0)
        return 1 - dice.mean()

class ComboLoss(nn.Module):
    def __init__(self, class_weights: list[float], num_classes: int, aux_weight: float,
                 dice_include_bg: bool = True):
        super().__init__()
        w = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
        self.ce = nn.CrossEntropyLoss(weight=w)
        self.dice = DiceLoss(num_classes, include_background=dice_include_bg)
        self.aux_weight = aux_weight
    def forward(self, out: dict, target: torch.Tensor) -> torch.Tensor:
        loss = self.ce(out["out"], target) + self.dice(out["out"], target)
        if out.get("aux") is not None:
            loss = loss + self.aux_weight * self.ce(out["aux"], target)
        return loss

criterion = ComboLoss(class_weights[CONFIG["WEIGHT_TYPE"]], CONFIG["NUM_CLASSES"],
                      CONFIG["AUX_WEIGHT"], dice_include_bg=CONFIG["DICE_INCLUDE_BG"])
optimizer = torch.optim.AdamW([
    {"params": model.backbone.parameters(),       "lr": CONFIG["LR_BACKBONE"]},
    {"params": model.classifier.parameters(),     "lr": CONFIG["LR_HEAD"]},
    {"params": model.aux_classifier.parameters(), "lr": CONFIG["LR_HEAD"]},
], weight_decay=CONFIG["WEIGHT_DECAY"])

total_steps = CONFIG["EPOCHS"] * len(loaders["train"])
warmup_steps = int(CONFIG["WARMUP_FRAC"] * total_steps)
def lr_factor(step: int) -> float:
    if step < warmup_steps: return (step + 1) / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1 + math.cos(math.pi * progress))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
scaler = torch.amp.GradScaler("cuda", enabled=(CONFIG["USE_AMP"] and DEVICE == "cuda"))
print(f"loss: Dice + weighted CE (+{CONFIG['AUX_WEIGHT']} aux) | AdamW | warmup: cosine | "
      f"steps: {total_steps} (warmup {warmup_steps})")

## 7. Training loop 

In [ ]:
print("Training configuration:")
for k, v in CONFIG.items(): print(f"  {k:12}: {v}")

def train_one_epoch() -> float:
    model.train(); running = n = 0
    for images, masks, _ in tqdm(loaders["train"], desc="train", leave=False):
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        with torch.amp.autocast("cuda", enabled=(CONFIG["USE_AMP"] and DEVICE == "cuda")):
            loss = criterion(model(images), masks)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update(); scheduler.step()
        running += loss.item() * images.size(0); n += images.size(0)
    return running / n

history = {"train_loss": [], "val_loss": [], "val_mIoU": [], "val_fg_mIoU": []}
best_fg_mIoU, best_epoch = -1.0, -1
ckpt_path = os.path.join(CONFIG["OUT_DIR"], f"{CONFIG['MODEL_NAME']}_best.pt")
start = time.time()

for epoch in range(1, CONFIG["EPOCHS"] + 1):
    train_loss = train_one_epoch()
    val_cm, val_loss = evaluate(model, loaders["val"], criterion)
    val_mIoU, val_fg_mIoU = val_cm.mean_iou, val_cm.foreground_mean_iou
    for key, val in zip(history, (train_loss, val_loss, val_mIoU, val_fg_mIoU)): history[key].append(val)
    print(f"epoch {epoch:02d} | train_loss {train_loss:.4f} | val_loss {val_loss:.4f} | "
          f"val_mIoU {val_mIoU:.4f} | val_fg_mIoU {val_fg_mIoU:.4f}")
    if val_fg_mIoU > best_fg_mIoU:                      # select on foreground mIoU
        best_fg_mIoU, best_epoch = val_fg_mIoU, epoch
        torch.save(model.state_dict(), ckpt_path)

train_minutes = (time.time() - start) / 60
print(f"\nDONE in {train_minutes:.1f} min | best val foreground mIoU {best_fg_mIoU:.4f} @ epoch {best_epoch}")

### Training curves loss & validation mIoU (overall vs foreground)

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(epochs, history["train_loss"], label="train")
ax[0].plot(epochs, history["val_loss"], label="val")
ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(epochs, history["val_mIoU"], label="overall mIoU", color="#4C78A8")
ax[1].plot(epochs, history["val_fg_mIoU"], label="foreground mIoU", color="#54A24B")
ax[1].axvline(best_epoch, ls="--", c="grey")
ax[1].set_title(f"Validation mIoU (best fg {best_fg_mIoU:.3f} @ {best_epoch})"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout(); plt.show()

## 8. Test-set evaluation

In [ ]:
model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
test_cm, _ = evaluate(model, loaders["test"])
names = [CONFIG["CLASS_NAMES"][k] for k in range(CONFIG["NUM_CLASSES"])]

per_class = pd.DataFrame({"class": names,
                          "IoU":  np.round(test_cm.per_class_iou, 4),
                          "Dice": np.round(test_cm.per_class_dice, 4)})
print(per_class.to_string(index=False))
print(f"\nmIoU (all 3)      : {test_cm.mean_iou:.4f}")
print(f"mIoU (foreground) : {test_cm.foreground_mean_iou:.4f}")
print(f"mean Dice (F1)    : {test_cm.mean_dice:.4f}")
print(f"pixel accuracy    : {test_cm.pixel_accuracy:.4f}")
print(f"mean pixel acc    : {test_cm.mean_pixel_accuracy:.4f}")

### Pixel-level confusion matrix

In [ ]:
cm = test_cm.matrix
cm_norm = cm / cm.sum(1, keepdims=True).clip(min=1)
fig, ax = plt.subplots(figsize=(5.2, 4.4))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(names))); ax.set_yticks(range(len(names)))
ax.set_xticklabels(names, rotation=30, ha="right"); ax.set_yticklabels(names)
ax.set_xlabel("predicted"); ax.set_ylabel("ground truth"); ax.set_title("Confusion matrix (row-normalized)")
for i in range(len(names)):
    for j in range(len(names)):
        ax.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center",
                color="white" if cm_norm[i, j] > 0.5 else "black", fontsize=9)
plt.colorbar(im, fraction=0.046); plt.tight_layout(); plt.show()

## 9. Error analysis

In [ ]:
@torch.no_grad()
def predict(filename: str) -> np.ndarray:
    model.eval()
    x = eval_tf(image=load_rgb(filename), mask=build_label(filename))["image"].unsqueeze(0).to(DEVICE)
    return model(x)["out"].argmax(1)[0].cpu().numpy()

def image_fg_iou(filename: str) -> tuple[float, np.ndarray]:
    pred = predict(filename)
    cm = ConfusionMatrix(CONFIG["NUM_CLASSES"])
    cm.update(torch.from_numpy(pred).long(), torch.from_numpy(build_label(filename)).long())
    present = cm.matrix.sum(1) > 0                       
    fg = [c for c in (1, 2) if present[c]]
    ious = cm.per_class_iou
    return (float(ious[fg].mean()) if fg else float(ious[present].mean())), pred

scored = [(fn, *image_fg_iou(fn)) for fn in tqdm(split["test"], desc="scoring test")]
scored.sort(key=lambda t: t[1])
def prediction_column(caption: str, fn: str, pred: np.ndarray):
    img = load_rgb(fn)
    return (caption, [img, overlay(img, build_label(fn)), overlay(img, pred)])

worst = scored[:5]
print("worst 5 test images by per-image foreground IoU:")
for fn, s, _ in worst: print(f"  {s:.3f}  {fn}")
plot_grid([prediction_column(f"{fn[:12]} {s:.2f}", fn, pred) for fn, s, pred in worst],
          ["image", "truth", "pred"], "Worst 5 test cases — red = dark, green = light")

### Qualitative predictions across the IoU range (worst -> best)

In [ ]:
n = len(scored) 
spread = [("worst", scored[0]), ("p25", scored[n // 4]), ("median", scored[n // 2]),
          ("p75", scored[3 * n // 4]), ("best", scored[-1])]
plot_grid([prediction_column(f"{label} {s:.2f}", fn, pred) for label, (fn, s, pred) in spread],
          ["image", "truth", "pred"], "Predictions across the IoU range (worst -> best)")

### Most-confused class pair

In [ ]:
off_diag = cm.copy(); np.fill_diagonal(off_diag, 0)
i, j = np.unravel_index(off_diag.argmax(), off_diag.shape)
print(f"Most confused: true '{names[i]}' predicted as '{names[j]}' "
      f"({off_diag[i, j]:,} px, {100 * off_diag[i, j] / cm.sum():.2f}% of all pixels)")

## 10. Summary

In [ ]:
results_path = find_artifact("results.json") or os.path.join(CONFIG["OUT_DIR"], "results.json")
results = json.load(open(results_path)) if os.path.isfile(results_path) else {"models": {}}
results.setdefault("models", {})[CONFIG["MODEL_NAME"]] = {
    "mIoU": test_cm.mean_iou, "foreground_mIoU": test_cm.foreground_mean_iou,
    "mDice": test_cm.mean_dice,
    "pixel_acc": test_cm.pixel_accuracy, "mean_pixel_acc": test_cm.mean_pixel_accuracy,
    "per_class_iou":  {names[k]: float(test_cm.per_class_iou[k]) for k in range(len(names))},
    "per_class_dice": {names[k]: float(test_cm.per_class_dice[k]) for k in range(len(names))},
    "selection_metric": "val_foreground_mIoU",
    "best_epoch": best_epoch, "epochs_run": CONFIG["EPOCHS"], "train_minutes": round(train_minutes, 1),
    "config": {k: CONFIG[k] for k in ("MODEL_NAME", "IMG_SIZE", "BATCH_SIZE", "EPOCHS",
               "LR_BACKBONE", "LR_HEAD", "WEIGHT_DECAY", "AUX_WEIGHT", "WEIGHT_TYPE", "DICE_INCLUDE_BG")},
}
out_path = os.path.join(CONFIG["OUT_DIR"], "results.json")
with open(out_path, "w") as f: json.dump(results, f, indent=2)
print("saved", out_path)
print(json.dumps(results["models"][CONFIG["MODEL_NAME"]], indent=2))